In [3]:
#!/usr/bin/env python3
# mmca_theta_del_sweep_parallel.py
"""
并行版本：使用 joblib 并行扫描 theta_del
"""

import os
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

# ----------------------- hypergraph generator -----------------------
def generate_hypergraph_H_n_m_3(n, kappa, seed=None, batch_size=3000):
    """Generate random 3-uniform hypergraph as ndarray shape (m,3). Uses RNG from seed if provided."""
    rng = np.random.RandomState(seed) if seed is not None else np.random
    d = 3
    m = int(round(n * kappa / d))
    max_possible = math.comb(n, d)
    if m > max_possible:
        raise ValueError("m > C(n,3)")
    nodes = np.arange(n)
    edges_set = set()
    while len(edges_set) < m:
        tris = rng.choice(nodes, size=(batch_size, d), replace=True)
        for i in range(tris.shape[0]):
            row = tris[i]
            if len(np.unique(row)) < d:
                row = rng.choice(nodes, size=(d,), replace=False)
            tris[i] = np.sort(row)
        for row in tris:
            if len(edges_set) >= m:
                break
            edges_set.add(tuple(row.tolist()))
    hyperedges = np.array(list(edges_set), dtype=int)
    hyperedges.sort(axis=1)
    return hyperedges

# ----------------------- MMCA with edge deletion -----------------------
class MMCAWithDelete:
    def __init__(self, hyperedges, n):
        self.hyperedges = np.asarray(hyperedges, dtype=int)
        self.n = n
        self.m = self.hyperedges.shape[0]
        # endpoints
        self.edge_i = self.hyperedges[:,0].copy()
        self.edge_j = self.hyperedges[:,1].copy()
        self.edge_k = self.hyperedges[:,2].copy()
        # node incidence list: for each node a list of (edge_idx, pos)
        self.node_edge_pos = [[] for _ in range(n)]
        for ei in range(self.m):
            a,b,c = self.hyperedges[ei]
            self.node_edge_pos[a].append((ei,0))
            self.node_edge_pos[b].append((ei,1))
            self.node_edge_pos[c].append((ei,2))

    def _compute_edge_products(self, p):
        """Compute p_j * p_k etc for all edges (vectorized)."""
        p_i = p[self.edge_i]; p_j = p[self.edge_j]; p_k = p[self.edge_k]
        return (p_j * p_k, p_i * p_k, p_i * p_j)

    def run_until_converged_with_deletion(self,
                                          beta_d, mu, eta, gamma,
                                          theta_del,
                                          T_max=500, tol=1e-9,
                                          init_frac=0.01,
                                          verbose=False):
        """执行带删除机制的MMCA迭代直到收敛"""
        n = self.n
        m = self.m
        p = np.full(n, init_frac, dtype=float)
        Theta = np.ones(m, dtype=float)
        active = np.ones(m, dtype=bool)  # True = edge active (not deleted)

        # histories (optional small lists)
        rho_hist = []
        theta_mean_hist = []
        iters = 0

        for it in range(T_max):
            rho_hist.append(p.mean()) 
            theta_mean_hist.append(Theta[active].mean() if active.any() else 0.0)

            # edge-products
            prod_pos0, prod_pos1, prod_pos2 = self._compute_edge_products(p)

            # compute Q_i = product_{e in E_i, active} (1 - Theta_e * beta * prod_others)
            Q = np.ones(n, dtype=float)
            for node in range(n):
                prod_val = 1.0
                for (ei, pos) in self.node_edge_pos[node]:
                    if not active[ei]:
                        continue

                    if pos == 0:
                        prod_oth = prod_pos0[ei]
                    elif pos == 1:
                        prod_oth = prod_pos1[ei]
                    else:
                        prod_oth = prod_pos2[ei]
                    prod_val *= (1.0 - Theta[ei] * beta_d * prod_oth)
                Q[node] = prod_val

            # MMCA update
            p_next = (1.0 - p) * (1.0 - Q) + (1.0 - mu) * p

            # Theta update (for active edges; inactive remain 0)
            p_i = p[self.edge_i]; p_j = p[self.edge_j]; p_k = p[self.edge_k]
            Phi = p_i * p_j * p_k
            Theta_next = Theta * (1.0 - eta * Phi) + gamma * (1.0 - Theta)
            Theta_next = np.clip(Theta_next, 0.0, 1.0)

            # deletion rule (permanent): if Theta_next < theta_del AND edge active -> delete it
            to_delete = np.where((Theta_next < theta_del) & (active))[0]
            if to_delete.size > 0:
                active[to_delete] = False
                Theta_next[to_delete] = 0.0

            # check convergence
            dp = np.max(np.abs(p_next - p))
            dTheta = np.max(np.abs(Theta_next - Theta))
            p = p_next
            Theta = Theta_next
            iters = it + 1

            if dp < tol and dTheta < tol:
                break

        rho_final = float(p.mean())
        num_deleted = int((~active).sum())
        f_del = num_deleted / float(m)
        return {
            'rho_final': rho_final,
            'f_del': f_del,
            'num_deleted': num_deleted,
            'iters': iters,
            'rho_hist': np.array(rho_hist),
            'theta_mean_hist': np.array(theta_mean_hist),
            'active_mask': active.copy()
        }

# ----------------------- 并行任务函数 -----------------------
def run_single_simulation(params):
    """
    单个模拟任务的并行执行函数
    """
    (theta_del, r, n, kappa, beta, mu, eta_global, gamma, 
     T_max, tol, init_frac, seed_base, reuse_graph) = params
    
    # 生成随机种子
    gen_seed = None if seed_base is None else (seed_base + r if not reuse_graph else seed_base)
    
    # 生成超图并运行模拟
    hyperedges = generate_hypergraph_H_n_m_3(n=n, kappa=kappa, seed=gen_seed)
    model = MMCAWithDelete(hyperedges, n)
    res = model.run_until_converged_with_deletion(
        beta_d=beta, mu=mu, eta=eta_global, gamma=gamma,
        theta_del=theta_del, T_max=T_max, tol=tol, init_frac=init_frac, verbose=False
    )
    
    # 返回结果
    return {
        'theta_del': theta_del,
        'run': r,
        'rho_final': res['rho_final'],
        'f_del': res['f_del'],
        'num_deleted': res['num_deleted'],
        'iters': res['iters']
    }

# ----------------------- 并行扫描驱动函数 -----------------------
def sweep_theta_del_parallel(n, kappa,
                            beta, mu, eta_global, gamma,
                            theta_del_list,
                            R=10,
                            reuse_graph=False,
                            T_max=1000,
                            tol=1e-8,
                            init_frac=0.01,
                            out_dir="out_theta_del_parallel",
                            seed_base=12345,
                            verbose=False,
                            n_jobs=-1):
    """
    并行版本：使用 joblib 并行执行所有模拟
    """
    os.makedirs(out_dir, exist_ok=True)
    
    # 准备所有参数组合
    print("准备参数组合...")
    param_combinations = []
    for theta_del in theta_del_list:
        for r in range(R):
            params = (theta_del, r, n, kappa, beta, mu, eta_global, gamma, 
                     T_max, tol, init_frac, seed_base, reuse_graph)
            param_combinations.append(params)
    
    print(f"总共 {len(param_combinations)} 个参数组合")
    
    # 使用 joblib 并行执行
    print("开始并行计算...")
    results = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_single_simulation)(params) 
        for params in param_combinations
    )
    
    # 收集结果
    print("收集结果...")
    records = list(results)
    
    # 创建 DataFrame
    df_records = pd.DataFrame(records)
    df_summary = df_records.groupby('theta_del').agg(
        mean_rho=('rho_final', 'mean'),
        std_rho=('rho_final', 'std'),
        mean_fdel=('f_del', 'mean'),
        std_fdel=('f_del', 'std'),
        R=('run', 'count')
    ).reset_index().sort_values('theta_del')

    # 保存输出
    summary_fn = os.path.join(out_dir, "theta_del_summary.csv")
    df_summary.to_csv(summary_fn, index=False, float_format="%.6f")

    perrun_fn = os.path.join(out_dir, "theta_del_perrun.csv")
    df_records.to_csv(perrun_fn, index=False, float_format="%.6f")

    # 绘图
    plt.figure(figsize=(8,4.5))
    plt.errorbar(df_summary['theta_del'], df_summary['mean_rho'], yerr=df_summary['std_rho'],
                 fmt='-o', capsize=3, label=r'$\rho^*$')
    plt.xlabel(r'$\theta_{\rm del}$')
    plt.ylabel(r'Steady infected fraction $\rho^*$')
    plt.title(r'$\rho^*$ vs self-isolation threshold $\theta_{\rm del}$')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "rho_vs_theta_del.png"), dpi=300)

    plt.figure(figsize=(8,4.5))
    plt.errorbar(df_summary['theta_del'], df_summary['mean_fdel'], yerr=df_summary['std_fdel'],
                 fmt='-s', capsize=3, color='C1', label='f_del')
    plt.xlabel(r'$\theta_{\rm del}$')
    plt.ylabel('Fraction of edges deleted $f_{del}$')
    plt.title('Deleted-edge fraction vs threshold')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "fdel_vs_theta_del.png"), dpi=300)

    return df_summary, df_records

# --------------------------- Parameters ---------------------------
n = 1500
kappa = 6.0
beta = 0.2
mu = 0.1
eta_global = 0.6
gamma = 0.02

theta_del_list = np.linspace(0.0, 0.4, 9)  # 0,0.05,...,0.4
R = 15                # repeats per theta_del
reuse_graph = False   # True -> reuse same hypergraph for all repeats
T_max = 500
tol = 1e-6
init_frac = 0.3
out_dir = "out_theta_del_parallel"
seed_base = 12345
verbose = True
n_jobs = 17  # 使用所有可用的CPU核心，可以改为具体数字

# --------------------------- Run sweep ---------------------------
if __name__ == "__main__":
    t0 = time.time()
    df_summary, df_records = sweep_theta_del_parallel(
        n=n, kappa=kappa, beta=beta, mu=mu, eta_global=eta_global, gamma=gamma,
        theta_del_list=theta_del_list, R=R, reuse_graph=reuse_graph,
        T_max=T_max, tol=tol, init_frac=init_frac,
        out_dir=out_dir, seed_base=seed_base, verbose=verbose, n_jobs=n_jobs
    )
    print("Saved summary to:", os.path.join(out_dir, "theta_del_summary.csv"))
    print("Saved per-run to:", os.path.join(out_dir, "theta_del_perrun.csv"))
    print("Done. Elapsed: {:.1f}s".format(time.time() - t0))

准备参数组合...
总共 135 个参数组合
开始并行计算...


[Parallel(n_jobs=17)]: Using backend LokyBackend with 17 concurrent workers.
[Parallel(n_jobs=17)]: Done   7 tasks      | elapsed:    3.2s
[Parallel(n_jobs=17)]: Done  16 tasks      | elapsed:    3.3s
[Parallel(n_jobs=17)]: Done  27 tasks      | elapsed:    6.0s
[Parallel(n_jobs=17)]: Done  38 tasks      | elapsed:    8.2s
[Parallel(n_jobs=17)]: Done  51 tasks      | elapsed:    8.9s
[Parallel(n_jobs=17)]: Done  64 tasks      | elapsed:   10.5s
[Parallel(n_jobs=17)]: Done  79 tasks      | elapsed:   11.5s
[Parallel(n_jobs=17)]: Done  94 tasks      | elapsed:   12.4s
[Parallel(n_jobs=17)]: Done 116 out of 135 | elapsed:   13.3s remaining:    2.1s


KeyboardInterrupt: 

准备参数组合...
总共 1860 个参数组合
theta_del 值: [0.0, 0.1, 0.2, 0.3]
beta 值: 31 个点
每个参数组合重复次数: 15
开始并行计算...


[Parallel(n_jobs=17)]: Using backend LokyBackend with 17 concurrent workers.
[Parallel(n_jobs=17)]: Done   7 tasks      | elapsed:    1.0s
[Parallel(n_jobs=17)]: Done  16 tasks      | elapsed:    1.0s
[Parallel(n_jobs=17)]: Done  27 tasks      | elapsed:    1.4s
[Parallel(n_jobs=17)]: Done  38 tasks      | elapsed:    1.8s
[Parallel(n_jobs=17)]: Done  51 tasks      | elapsed:    1.9s
[Parallel(n_jobs=17)]: Done  64 tasks      | elapsed:    2.3s
[Parallel(n_jobs=17)]: Done  79 tasks      | elapsed:    2.8s
[Parallel(n_jobs=17)]: Done  94 tasks      | elapsed:    3.2s
[Parallel(n_jobs=17)]: Done 111 tasks      | elapsed:    3.6s
[Parallel(n_jobs=17)]: Done 128 tasks      | elapsed:    4.0s
[Parallel(n_jobs=17)]: Done 147 tasks      | elapsed:    4.6s
[Parallel(n_jobs=17)]: Done 166 tasks      | elapsed:    5.1s
[Parallel(n_jobs=17)]: Done 187 tasks      | elapsed:    5.6s
[Parallel(n_jobs=17)]: Done 208 tasks      | elapsed:    6.3s
[Parallel(n_jobs=17)]: Done 231 tasks      | elapsed:  

KeyboardInterrupt: 